# Winograd reduced-precision convolution: measured GPU study

This notebook reads the PyTorch NCHW benchmark for F(2x2, 3x3) Winograd, covering FP32, FP16, BF16, transform-domain INT8, and stage-wise requantization on representative ResNet-18 layer shapes.

A configuration is worthwhile only when it is faster than its matching PyTorch/cuDNN precision baseline **and** its maximum absolute error against the FP64 oracle is within the recorded budget (default 0.1). Runtime is not treated as energy or power.

The checked-in sweep contains 72 Winograd configurations on an RTX 4060 Laptop GPU, with 100 CUDA-event samples and 5 repeats per configuration. **None was both faster than its matching baseline and within the error budget.** Inputs are seeded standard-normal tensors: this is a synthetic layerwise study, not a ResNet accuracy test. INT8 paths are quantization simulations, not a dedicated accelerator kernel.

In [ ]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results" / "results.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

with (PROJECT_ROOT / "results" / "results.csv").open(newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
print(f"Loaded {len(rows)} benchmark rows")
print(f"GPU: {rows[0]['device']}; driver: {rows[0]['driver']}; PyTorch: {rows[0]['torch_version']}; CUDA: {rows[0]['cuda_runtime']}")
print(f"CUDA event samples/repeats: {rows[0]['n_runs']}/{rows[0]['repeats']}; error budget: {rows[0]['error_budget']}")

In [ ]:
from collections import defaultdict
import statistics

by_precision = defaultdict(list)
for row in rows:
    if row["method"] == "winograd":
        by_precision[row["precision"]].append(row)

print("precision       N  mean speedup  best speedup      within budget  worthwhile")
for precision, group in by_precision.items():
    speedups = [float(row["speedup_vs_precision_baseline"]) for row in group]
    best = max(group, key=lambda row: float(row["speedup_vs_precision_baseline"]))
    accurate = sum(float(row["max_abs_error"]) <= float(row["error_budget"]) for row in group)
    worthwhile = sum(row["worthwhile_at_budget"].lower() == "true" for row in group)
    print(f"{precision:15s} {len(group):2d} {statistics.mean(speedups):12.3f}x  "
          f"{float(best['speedup_vs_precision_baseline']):8.3f}x ({best['layer']}, N={best['batch']})  "
          f"{accurate:3d}/{len(group):<3d}          {worthwhile}")

In [ ]:
import torch
import torch.nn.functional as functional
from src.torch_winograd import winograd_conv2d

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = torch.Generator().manual_seed(7)
x = torch.randn((2, 3, 9, 11), generator=generator).to(device)
weight = torch.randn((5, 3, 3, 3), generator=generator).to(device)
actual = winograd_conv2d(x, weight, padding=1)
oracle = functional.conv2d(x.cpu().double(), weight.cpu().double(), padding=1).float()
max_error = (actual.cpu() - oracle).abs().max().item()
print(f"{device}: batched multi-channel FP32 vs FP64 oracle max error = {max_error:.3e}")
assert max_error < 1e-4

In [ ]:
plot_names = [
    "crossover_speedup.png",
    "accuracy_runtime_tradeoff.png",
    "error_vs_method.png",
    "mse_vs_method.png",
    "runtime_vs_inputsize.png",
]
for name in plot_names:
    path = PROJECT_ROOT / "results" / name
    print(f"{name}: {path} (exists={path.exists()})")

## Conclusion

For this unfused PyTorch implementation and tested RTX 4060 layer/batch sweep, none of the 72 Winograd configurations was worthwhile under the stated criterion. The 16-product arithmetic count alone does not overcome transform, allocation, and launch overhead. The result is not a universal statement about Winograd: a fused CUDA/Triton implementation and calibrated real activations would be needed to test a hardware-level crossover. No energy or model-accuracy claim is supported.

Reproduce with `python main.py --runs 100 --repeats 5`; rebuild plots and report with `python main.py --plot-only`. See `../results/project_presentation_report.md` and `../README.md` for methodology and references.